<a href="https://colab.research.google.com/github/FarabiOnAMission/Machine-Learning-Stuffs/blob/main/Dimensionality_Reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import numpy as np

In [16]:
class PCA:
  def __init__(self, n_components):
    self.n_components = n_components
    self.components = None
    self.mean = None
    self.eigenvalues = None
    self.explained_variance_ratio = None

  def fit(self,X):
    self.mean = np.mean(X, axis = 0)
    X_centered = X - self.mean

    cov_matrix = np.cov(X_centered, rowvar = False)
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

    sorted_idx = np.argsort(eigenvalues)[::-1] #sorts the eigenvalues descending
    eigenvalues = eigenvalues[sorted_idx]
    eigenvectors = eigenvectors[:, sorted_idx]

    self.components = eigenvectors[:, :self.n_components] #slices the top k eigenvectors
    self.eigenvalues = eigenvalues[:self.n_components] #slices the top k eigenvalues
    total_var = np.sum(eigenvalues) #sums up all the eigenvalues to later use it in information discovery
    self.explained_variance_ratio = self.eigenvalues / total_var #finds the total information we gain by using this matrix

  def transform(self, X):
    X_centered = X - self.mean
    return np.dot(X_centered, self.components)

  def fit_transform(self, X):
    self.fit(X)
    return self.transform(X)

In [14]:
np.random.seed(42)
n_samples = 500

t = np.random.uniform(0, 2 * np.pi, n_samples)
x1 = 3 * np.cos(t) + np.random.normal(0, 0.2, n_samples)
x2 = 3 * np.sin(t) + np.random.normal(0, 0.2, n_samples)
x3 = 0.5 * x1 + 0.3 * x2 + np.random.normal(0, 0.1, n_samples)


X_synthetic = np.column_stack([x1, x2, x3])
pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_synthetic)
print(pca.explained_variance_ratio)

[0.59004014 0.40926775]


In [17]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X_mnist = mnist.data[:5000].astype(float)
y_mnist = mnist.target[:5000].astype(int)

pca_mnist = PCA(n_components=50)
X_pca50 = pca_mnist.fit_transform(X_mnist)
print(f"50 components capture {sum(pca_mnist.explained_variance_ratio):.2%} of variance")

pca_2d = PCA(n_components=2)
X_pca2d = pca_2d.fit_transform(X_mnist)
print(f"2 components capture {sum(pca_2d.explained_variance_ratio):.2%} of variance")

50 components capture 82.91% of variance
2 components capture 17.27% of variance


In [20]:
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.manifold import TSNE

sklearn_pca = SklearnPCA(n_components=2)
X_sklearn_pca = sklearn_pca.fit_transform(X_mnist)

print(f"\nOur PCA explained variance:     {pca_2d.explained_variance_ratio}")
print(f"Sklearn PCA explained variance: {sklearn_pca.explained_variance_ratio_}")

diff = np.abs(np.abs(X_pca2d) - np.abs(X_sklearn_pca))
print(f"Max absolute difference: {diff.max():.10f}")

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_mnist)
print(f"\nt-SNE output shape: {X_tsne.shape}")


Our PCA explained variance:     [0.09867566 0.07404546]
Sklearn PCA explained variance: [0.09867566 0.07404546]
Max absolute difference: 0.0250733103

t-SNE output shape: (5000, 2)
